# Current sheets in $J_\parallel^2$ and $J_\perp^2$ across the beta scan

One notebook for the four that ran the same analysis run by run (`Jpar_CSDetectionBeta0.0625`, `Jpar_CSDetectionBeta1`,
`Jpar_CSDetectionBeta16` and `Jpar_CSDetectionMethod5_Beta0.0625`, now in `archive/jpar_jperp_cs_detection/`).
For every run, at one time:

1. J from the Gaussian-filtered B; $J_\parallel^2 = (J \cdot B)^2/|B|^2$ and $J_\perp^2 = J^2 - J_\parallel^2$, both filtered again.
2. Current sheets where $J_\parallel^2$ (or $J_\perp^2$) is above a quantile, cleaned by opening and closing with a disk
   and dropping small regions; the maximum of each sheet, and the sheet thicknesses (diameter of the largest
   inscribed circle) with a lognormal fit.
3. The same thicknesses for the sheets of $|J_\perp|$ found by peak flooding (the former Method 5 notebook).
4. Maps of $J_\perp^2$, $J^2$ and $|J_\parallel|/|J|$, and the spectra of J, $J_\parallel$, $J_\perp$ and $J_{xy}$.

The choices that differed between the old notebooks are in `RUN_SETTINGS`.

In [ ]:
import csv

import numpy as np
import matplotlib.pyplot as plt

from pta import config
from pta.fields import current_density, parallel_perpendicular_squared
from pta.filtering import gaussian_fft_filter_2d
from pta.plotting import mark_cells, outline_mask, save_figure, show_field, use_style
from pta.sheets import detect_current_sheets, get_max_coords, quantile_mask, region_thicknesses, sheets_to_mask
from pta.spectra import parallel_perpendicular_spectra, summed_spectrum
from pta.statistics import fit_and_plot_pdf

use_style()
RESULTS = config.results_dir("jpar_jperp", "JparJperpCSDetection")

In [ ]:
# ---- runs and time ----
BETAS_P = [0.0625, 1, 16]             # proton beta of the runs (the beta_scan layout in pta.toml: beta_e = 1, B_rms = 0.24)
SNAPSHOT_TIME = 40

# ---- the choices that differed between the old per-run notebooks ----
# filter_fraction: width k0 of the Gaussian filters, as a fraction of the number of cells
# quantile_par, quantile_perp: J_par^2 or J_perp^2 above this quantile counts as current sheet
RUN_SETTINGS = {
    0.0625: dict(filter_fraction=0.2, quantile_par=0.8, quantile_perp=0.8),
    1: dict(filter_fraction=0.3, quantile_par=0.9, quantile_perp=0.8),
    16: dict(filter_fraction=0.2, quantile_par=0.8, quantile_perp=0.8),
}

# ---- cleaning the masks, in grid cells ----
SMOOTHING_RADIUS_CELLS = 5            # disk used to open and close the mask
MIN_AREA_CELLS = 300                  # regions with fewer cells are dropped

# ---- peak flooding of |J_perp| (the former Method 5 notebook) ----
FLOODING_GLOBAL_FRACTION = 0.2        # stop when the highest remaining peak is below this fraction of the maximum
FLOODING_LOCAL_FRACTION = 0.5         # a sheet is the connected region above this fraction of its peak

# ---- figures ----
PDF_BINS = 10
SPECTRUM_K_RANGE = (5e-2, 30)
SPECTRUM_POWER_RANGE = (1e-6, 1)

In [ ]:
def fields_at(beta_p):
    """J^2, J_par^2, J_perp^2 and |J_perp| of one run at SNAPSHOT_TIME (J from the filtered B; the direction
    along B from the unfiltered B), with the grid spacing, J and B for the spectra."""
    run = config.run("beta_scan", beta_p=beta_p)
    filter_width = RUN_SETTINGS[beta_p]["filter_fraction"] * run.shape[0]
    B = run.load_many(["Bx", "By", "Bz"], SNAPSHOT_TIME)
    J = current_density(*(gaussian_fft_filter_2d(component, k0=filter_width) for component in B), run.dx, run.dy)
    J_par_squared, J_perp_squared = parallel_perpendicular_squared(J, B)
    return dict(
        dx=run.dx, dy=run.dy, J=J, B=B,
        J_squared=J[0]**2 + J[1]**2 + J[2]**2,
        J_par_squared=gaussian_fft_filter_2d(J_par_squared, k0=filter_width),
        J_perp_squared=gaussian_fft_filter_2d(J_perp_squared, k0=filter_width),
        J_perp_magnitude=gaussian_fft_filter_2d(np.sqrt(np.abs(J_perp_squared)), k0=filter_width),
    )


def map_with_sheets(field, mask, dx, dy, title, name, maxima=None, outline_colour="yellow"):
    """The field with the outlines of the current sheets (and a cross at each sheet's maximum)."""
    fig, ax = plt.subplots(figsize=(10, 8))
    show_field(ax, field, dx, dy)
    outline_mask(ax, mask, dx, dy, color=outline_colour)
    if maxima is not None:
        mark_cells(ax, maxima, dx, dy)
    ax.set_title(title)
    save_figure(fig, RESULTS, name)
    plt.show()


def thickness_pdf(thicknesses, title, name):
    """Histogram and lognormal fit (loc = 0 as in the Paper 1 pipeline, mode marked) of sheet thicknesses in d_p;
    returns the statistics."""
    if len(thicknesses):
        print(f"  {len(thicknesses)} sheets: thickness mean {np.mean(thicknesses):.3f}, median {np.median(thicknesses):.3f}, "
              f"min {np.min(thicknesses):.3f}, max {np.max(thicknesses):.3f} d_p")
    fig, ax = plt.subplots()
    statistics = fit_and_plot_pdf(ax, thicknesses, title, bins=PDF_BINS, ylabel=True, mark="mode")
    save_figure(fig, RESULTS, name)
    plt.show()
    return statistics

In [ ]:
summary = []                                    # the thickness statistics of every run and kind of sheet

for beta_p in BETAS_P:
    fields = fields_at(beta_p)
    dx, dy, run_label = fields["dx"], fields["dy"], f"beta_p{beta_p}"
    settings = RUN_SETTINGS[beta_p]
    when = rf"$\beta_p = {beta_p}$, t = {SNAPSHOT_TIME}"

    # ---- sheets where J_par^2 and J_perp^2 are above their quantile ----
    masks = {}
    for component, symbol in (("par", r"\parallel"), ("perp", r"\perp")):
        field = fields[f"J_{component}_squared"]
        mask = quantile_mask(field, settings[f"quantile_{component}"], SMOOTHING_RADIUS_CELLS, MIN_AREA_CELLS)
        masks[component] = mask
        maxima, _ = get_max_coords(mask, field)
        print(f"beta_p = {beta_p}, J_{component}^2 above the {settings[f'quantile_{component}']} quantile: {len(maxima)} regions")
        map_with_sheets(field, mask, dx, dy, rf"$J_{symbol}^2$ and its current sheets, " + when,
                        f"J_{component}_squared_sheets_{run_label}")
        map_with_sheets(field, mask, dx, dy, rf"Maximum of $J_{symbol}^2$ in each sheet, " + when,
                        f"J_{component}_squared_maxima_{run_label}", maxima=maxima, outline_colour="white")
        statistics = thickness_pdf(region_thicknesses(mask, step=dx), rf"$J_{symbol}^2$ sheets, " + when,
                                   f"J_{component}_thickness_pdf_{run_label}")
        summary.append(dict(beta_p=beta_p, sheets=f"J_{component}^2 above quantile", **statistics))

    # ---- J_perp^2 and J^2 with the J_par^2 sheets, and |J_par| / |J| ----
    map_with_sheets(fields["J_perp_squared"], masks["par"], dx, dy, r"$J_\perp^2$ with the $J_\parallel^2$ sheets, " + when,
                    f"J_perp_squared_with_J_par_sheets_{run_label}")
    map_with_sheets(fields["J_squared"], masks["par"], dx, dy, r"$J^2$ with the $J_\parallel^2$ sheets, " + when,
                    f"J_squared_with_J_par_sheets_{run_label}")
    fig, ax = plt.subplots(figsize=(10, 8))
    J_par_fraction = np.sqrt(fields["J_par_squared"] / (fields["J_par_squared"] + fields["J_perp_squared"]))
    show_field(ax, J_par_fraction, dx, dy, label=r"$|J_\parallel| / |J|$")
    ax.set_title(r"$|J_\parallel| / |J|$, " + when)
    save_figure(fig, RESULTS, f"J_par_fraction_{run_label}")
    plt.show()

    # ---- sheets of |J_perp| by peak flooding (the former Method 5 notebook) ----
    magnitude = fields["J_perp_magnitude"]
    sheets = detect_current_sheets(magnitude, global_threshold=FLOODING_GLOBAL_FRACTION * magnitude.max(),
                                   local_fraction=FLOODING_LOCAL_FRACTION)
    flooding_mask = sheets_to_mask(sheets, magnitude.shape)
    print(f"beta_p = {beta_p}, |J_perp| by peak flooding: {len(sheets)} sheets")
    map_with_sheets(magnitude, flooding_mask, dx, dy, r"$|J_\perp|$ and its sheets by peak flooding, " + when,
                    f"J_perp_magnitude_flooding_sheets_{run_label}")
    statistics = thickness_pdf(region_thicknesses(flooding_mask, step=dx), r"$|J_\perp|$ sheets by peak flooding, " + when,
                               f"J_perp_flooding_thickness_pdf_{run_label}")
    summary.append(dict(beta_p=beta_p, sheets="|J_perp| peak flooding", **statistics))

    # ---- spectra of J, J_par, J_perp and J_xy (J from the filtered B) ----
    k, total, parallel, perpendicular = parallel_perpendicular_spectra(fields["J"], fields["B"], dx, dy)
    _, in_plane = summed_spectrum(fields["J"][:2], dx, dy)
    fig, ax = plt.subplots()
    for spectrum, name in ((total, "J"), (parallel, r"J_\parallel"), (perpendicular, r"J_\perp"), (in_plane, "J_{xy}")):
        ax.loglog(k, spectrum, label=rf"${name}$ spectrum")
    ax.set_xlabel(r"$k$")
    ax.set_ylabel(r"$P(k)$")
    ax.set_xlim(*SPECTRUM_K_RANGE)
    ax.set_ylim(*SPECTRUM_POWER_RANGE)
    ax.set_title(when)
    ax.legend()
    save_figure(fig, RESULTS, f"J_spectra_{run_label}")
    plt.show()

In [ ]:
# The thickness statistics of every run, in one table
columns = ["beta_p", "sheets", "n", "median_lognorm", "mode_lognorm", "sigma_lognorm", "loc_lognorm", "mean", "median_emp"]
with open(RESULTS / "thickness_statistics.csv", "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(summary)
print(f"saved {RESULTS / 'thickness_statistics.csv'}")